In [1]:
import numpy as np

class LinUCB:
    def __init__(self, n_actions, context_dim, alpha=1.0):
        self.n_actions = n_actions
        self.alpha = alpha
        
        # Para cada ação: matriz A e vetor b
        self.A = [np.eye(context_dim) for _ in range(n_actions)]
        self.b = [np.zeros((context_dim, 1)) for _ in range(n_actions)]

    def select_action(self, context):
        context = context.reshape(-1, 1)
        p_values = []

        for a in range(self.n_actions):
            A_inv = np.linalg.inv(self.A[a])
            theta = A_inv @ self.b[a]

            # estimativa de reward
            mean = float(theta.T @ context)

            # incerteza (exploração)
            uncertainty = float(
                self.alpha * np.sqrt(context.T @ A_inv @ context)
            )

            p = mean + uncertainty
            p_values.append(p)

        return np.argmax(p_values)

    def update(self, action, context, reward):
        context = context.reshape(-1, 1)

        self.A[action] += context @ context.T
        self.b[action] += reward * context

In [3]:
# ações do advisor
actions = ["ask", "recommend", "compare", "explore"]

bandit = LinUCB(n_actions=4, context_dim=3, alpha=0.5)


In [8]:
bandit.A

[array([[1., 0., 0.],
        [0., 1., 0.],
        [0., 0., 1.]]),
 array([[1., 0., 0.],
        [0., 1., 0.],
        [0., 0., 1.]]),
 array([[1., 0., 0.],
        [0., 1., 0.],
        [0., 0., 1.]]),
 array([[1., 0., 0.],
        [0., 1., 0.],
        [0., 0., 1.]])]

In [10]:
bandit.b

[array([[0.],
        [0.],
        [0.]]),
 array([[0.],
        [0.],
        [0.]]),
 array([[0.],
        [0.],
        [0.]]),
 array([[0.],
        [0.],
        [0.]])]

In [2]:

def simulate_user_feedback(action, context):
    # regra fake (só pra simular)
    if action == 0:  # ask
        return 1 if context[0] < 0.5 else 0
    elif action == 1:  # recommend
        return 1 if context[1] > 0.5 else 0
    elif action == 2:  # compare
        return 1 if context[2] > 0.5 else 0
    else:
        return np.random.rand()

# loop de interação
for t in range(1000):
    # contexto: [certeza, histórico, diversidade]
    context = np.random.rand(3)

    action = bandit.select_action(context)
    reward = simulate_user_feedback(action, context)

    bandit.update(action, context, reward)

# teste final
test_context = np.array([0.2, 0.8, 0.3])
best_action = bandit.select_action(test_context)

print("Melhor ação:", actions[best_action])

Melhor ação: recommend


/var/folders/px/1q7fhnbd7q51byjntbnvzsq40000gn/T/ipykernel_84553/2474090945.py:21: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  mean = float(theta.T @ context)
/var/folders/px/1q7fhnbd7q51byjntbnvzsq40000gn/T/ipykernel_84553/2474090945.py:24: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  uncertainty = float(
